<a href="https://colab.research.google.com/github/ingaxelga-web/HRAIE_tool/blob/main/Red_Neuronal_XOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 4. Experimentación comparando tres tasas de aprendizaje (η = 0.05, 0.50 y 2.00)
epochs = 8000
experimentos = [
    {
        'nombre': 'Experimento 1: Tasa Baja (η = 0.05)',
        'lr': 0.05,
        'color': '#D97706',
        'estilo': '--',
    },
    {
        'nombre': 'Experimento 2: Tasa Óptima (η = 0.50)',
        'lr': 0.50,
        'color': '#006633',
        'estilo': '-',
    },
    {
        'nombre': 'Experimento 3: Tasa Alta (η = 2.00)',
        'lr': 2.00,
        'color': '#1B3A2E',
        'estilo': '-.',
    },
]

historiales = {}
modelos = {}

for exp in experimentos:
  red = RedNeuronalXOR(input_dim=2, hidden_dim=4, output_dim=1, seed=42)
  perdidas = red.train(X, y, epochs=epochs, learning_rate=exp['lr'])
  historiales[exp['nombre']] = perdidas
  modelos[exp['nombre']] = red

  pred = red.forward(X)
  pred_bin = (pred >= 0.5).astype(int)
  acc = np.mean(pred_bin == y) * 100
  print(f"=== {exp['nombre']} ===")
  print(
      f'MSE Inicial: {perdidas[0]:.6f} | MSE Final: {perdidas[-1]:.6f} |'
      f' Exactitud: {acc:.1f}%'
  )
  for i in range(len(X)):
    print(
        f'  Patrón {X[i].astype(int)} -> Esperado: {int(y[i][0])} | Predicción:'
        f' {pred[i][0]:.4f} (Clase {pred_bin[i][0]})'
    )
  print()

NameError: name 'RedNeuronalXOR' is not defined

In [ ]:
# 5. Gráfica 1: Curvas de Pérdida (Loss vs. Épocas)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(
    historiales[experimentos[0]['nombre']],
    color='#D97706',
    lw=2,
    linestyle='--',
    label='Exp 1: η = 0.05',
)
axes[0].plot(
    historiales[experimentos[1]['nombre']],
    color='#006633',
    lw=2.2,
    label='Exp 2: η = 0.50',
)
axes[0].set_title(
    'Comparativa: Tasa Baja vs. Tasa Óptima',
    fontweight='bold',
    color='#1B3A2E',
)
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('Pérdida MSE (Loss)')
axes[0].grid(True, linestyle=':', alpha=0.6)
axes[0].legend()

for exp in experimentos:
  axes[1].plot(
      historiales[exp['nombre']],
      color=exp['color'],
      lw=2,
      linestyle=exp['estilo'],
      label=exp['nombre'],
  )
axes[1].set_title(
    'Dinámica Global de Convergencia (8,000 Épocas)',
    fontweight='bold',
    color='#1B3A2E',
)
axes[1].set_xlabel('Épocas')
axes[1].set_ylabel('Pérdida MSE (Loss)')
axes[1].grid(True, linestyle=':', alpha=0.6)
axes[1].legend()
plt.tight_layout()
plt.show()

# 6. Gráfica 2: Frontera de Decisión No Lineal en 2D aprendida por el MLP
modelo_opt = modelos['Experimento 2: Tasa Óptima (η = 0.50)']
x_grid = np.linspace(-0.2, 1.2, 200)
X1, X2 = np.meshgrid(x_grid, x_grid)
Z = modelo_opt.forward(np.c_[X1.ravel(), X2.ravel()]).reshape(X1.shape)

plt.figure(figsize=(6.5, 5))
contour = plt.contourf(X1, X2, Z, levels=40, cmap='YlGn', alpha=0.85)
plt.colorbar(contour, label='Probabilidad P(Y=1 | X)')
plt.contour(
    X1, X2, Z, levels=[0.5], colors='#1B3A2E', linewidths=2.2, linestyles='--'
)

for i in range(len(X)):
  if y[i] == 1:
    plt.scatter(
        X[i, 0],
        X[i, 1],
        color='#6AB023',
        s=160,
        edgecolors='#1B3A2E',
        lw=2,
        label='Clase 1 (XOR=1)' if i == 1 else '',
    )
  else:
    plt.scatter(
        X[i, 0],
        X[i, 1],
        color='#1B3A2E',
        s=150,
        marker='s',
        edgecolors='white',
        lw=1.5,
        label='Clase 0 (XOR=0)' if i == 0 else '',
    )
  plt.text(
      X[i, 0] + 0.04,
      X[i, 1] + 0.04,
      f'({int(X[i,0])},{int(X[i,1])})->{int(y[i,0])}',
      fontweight='bold',
      color='#1B3A2E',
  )

plt.title(
    'Frontera de Decisión No Lineal Aprendida por el MLP [2-4-1]',
    fontweight='bold',
    color='#1B3A2E',
)
plt.xlabel('Entrada x1')
plt.ylabel('Entrada x2')
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Funciones de activación y derivadas analíticas
def sigmoid(z):
  """Función de activación Sigmoide: σ(z) = 1 / (1 + e^(-z))"""
  return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def sigmoid_derivative(a):
  """Derivada de la sigmoide evaluada en la activación a: σ'(z) = a * (1 - a)"""
  return a * (1.0 - a)


# 3. Clase de la Red Neuronal Multicapa [2 - 4 - 1]
class RedNeuronalXOR:

  def __init__(self, input_dim=2, hidden_dim=4, output_dim=1, seed=42):
    np.random.seed(seed)
    self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
    self.b1 = np.zeros((1, hidden_dim))
    self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(
        2.0 / hidden_dim
    )
    self.b2 = np.zeros((1, output_dim))

  def forward(self, X):
    """Paso 1: Forward Pass (Propagación hacia adelante)"""
    self.Z1 = np.dot(X, self.W1) + self.b1
    self.A1 = sigmoid(self.Z1)
    self.Z2 = np.dot(self.A1, self.W2) + self.b2
    self.A2 = sigmoid(self.Z2)
    return self.A2

  def compute_loss(self, y_pred, y_true):
    """Paso 2: Cálculo de Error Cuadrático Medio (MSE Loss)"""
    return np.mean((y_pred - y_true) ** 2)

  def backward(self, X, y_true, learning_rate):
    """Paso 3: Retropropagación (Backpropagation) y Paso 4: Actualización de Pesos"""
    m = X.shape[0]
    # Gradiente en la capa de salida
    error_salida = self.A2 - y_true
    delta_salida = error_salida * sigmoid_derivative(self.A2)
    dW2 = np.dot(self.A1.T, delta_salida) / m
    db2 = np.sum(delta_salida, axis=0, keepdims=True) / m

    # Retropropagación hacia la capa oculta
    delta_oculta = np.dot(delta_salida, self.W2.T) * sigmoid_derivative(self.A1)
    dW1 = np.dot(X.T, delta_oculta) / m
    db1 = np.sum(delta_oculta, axis=0, keepdims=True) / m

    # Actualización de pesos por Descenso de Gradiente
    self.W2 -= learning_rate * dW2
    self.b2 -= learning_rate * db2
    self.W1 -= learning_rate * dW1
    self.b1 -= learning_rate * db1

  def train(self, X, y, epochs=8000, learning_rate=0.5):
    loss_history = []
    for epoch in range(epochs):
      y_pred = self.forward(X)
      loss = self.compute_loss(y_pred, y)
      loss_history.append(loss)
      self.backward(X, y, learning_rate)
    return loss_history


print("Arquitectura RedNeuronalXOR [2 - 4 - 1] definida correctamente.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Semilla fija para reproducibilidad de los experimentos
np.random.seed(42)

# 1. Definición del Dataset de la compuerta lógica XOR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)

y = np.array([[0], [1], [1], [0]], dtype=float)

print("Matriz de entradas X:\n", X)
print("Vector de salidas deseadas y:\n", y)